# Building a Synthetic Image Dataset with Sana

_Authored by: [Parag Ekbote](https://github.com/ParagEkbote)_ and [David Berenstein](https://github.com/davidberenstein1957)

In [ ]:
!pip install -q pruna metaflow datasets

In [ ]:
import random
from pathlib import Path

import torch
from PIL import Image

from metaflow import FlowSpec, step
from datasets import Dataset, load_dataset, Image as ImageFeature
from transformers import AutoProcessor, AutoModelForImageTextToText
from diffusers import SanaPipeline
from pruna import smash, SmashConfig
from huggingface_hub import HfApi, login # Import HfApi and login

# Ensure you have logged into Hugging Face Hub:
# From your terminal, run: huggingface-cli login
# And enter your Hugging Face token.

class SyntheticImageFlow(FlowSpec):
    """
    A Metaflow pipeline for generating synthetic image data through iterative
    prompt rewriting and image generation, evaluated by a vision-language model
    (Google Gemma-3-4b-it), and finally uploaded to Hugging Face Hub.
    """

    # Define flow parameters
    num_iterations = 3  # Number of iterations for prompt rewriting and image generation
    num_prompts_per_iteration = 50 # Number of images to generate per iteration
    top_k_prompts_for_next_iteration = 25 # Number of top-scoring prompts to carry over
    hf_dataset_name = "your-username/synthetic-image-dataset" # Replace with your HF username and desired dataset name

    @step
    def start(self):
        """
        Initializes the pipeline by loading a subset of a prompt-image preference dataset
        and setting up iteration tracking.
        """
        print("📥 Loading a subset of prompt-image preference dataset...")
        # Load a small subset for demonstration purposes
        dataset = load_dataset("data-is-better-together/open-image-preferences-v1", split="train[:1%]")
        self.pairs = [
            (ex["prompt"], ex["preferred"])
            for ex in dataset
            if ex["preferred"] in [0, 1]
        ]
        print(f"✅ Loaded {len(self.pairs)} prompt-image pairs.")

        self.iteration = 0
        self.all_generated_data = [] # Stores (prompt, image_path, score, iteration) for all runs
        self.next(self.select_prompts)

    @step
    def select_prompts(self):
        """
        Selects initial prompts from the loaded dataset. In subsequent iterations,
        this step will receive top-scoring prompts from the evaluation step.
        """
        if self.iteration == 0:
            print("🔍 Selecting initial prompts from preferred image pairs...")
            self.selected_prompts = [prompt for prompt, _ in self.pairs]
            # Limit initial prompts to match num_prompts_per_iteration for consistency
            self.selected_prompts = self.selected_prompts[:self.num_prompts_per_iteration]
            print(f"✅ {len(self.selected_prompts)} initial prompts selected.")
        else:
            print(f"🔍 Selecting top prompts for iteration {self.iteration}...")
            # self.selected_prompts will already be set by evaluate_images from previous iteration
            print(f"✅ {len(self.selected_prompts)} prompts selected for re-generation.")

        self.next(self.rewrite_prompts)

    @step
    def rewrite_prompts(self):
        """
        Rewrites the selected prompts by adding descriptive adjectives to introduce variety.
        This step helps explore different visual interpretations.
        """
        print(f"✍️ Rewriting prompts for variety (Iteration {self.iteration + 1}/{self.num_iterations})...")
        adjectives = ["surreal", "epic", "dreamy", "vibrant", "macro", "eerie", "futuristic", "ancient", "minimalist"]
        self.rewritten_prompts = [
            f"{random.choice(adjectives)} {prompt}" for prompt in self.selected_prompts
        ]
        print(f"✅ {len(self.rewritten_prompts)} prompts rewritten.")
        self.next(self.generate_images)

    @step
    def generate_images(self):
        """
        Generates images using the SanaPipeline and Pruna's smash() for optimization.
        Images are saved locally.
        """
        print(f"🖼️ Generating images using SanaPipeline + Pruna's smash() (Iteration {self.iteration + 1}/{self.num_iterations})...")

        # Load and optimize the pipeline once per step execution
        # Using a context manager for device placement is good practice
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {device}")

        # Initialize pipeline
        pipe = SanaPipeline.from_pretrained(
            "Efficient-Large-Model/Sana_600M_512px_diffusers",
            torch_dtype=torch.float16 if device.type == "cuda" else torch.float32 # Use float16 on GPU
        ).to(device)

        # Apply Pruna's smash for optimization
        smashed_pipe = smash(
            model=pipe,
            smash_config=SmashConfig(compilers=["diffusers2"]),
        )

        self.generated_this_iteration = [] # Stores (prompt, image_path) for current iteration
        output_dir = Path(f"outputs/iteration_{self.iteration}")
        output_dir.mkdir(parents=True, exist_ok=True)

        # Generate images for the rewritten prompts
        for i, prompt in enumerate(self.rewritten_prompts[:self.num_prompts_per_iteration]):
            try:
                # Generate image
                image = smashed_pipe(prompt).images[0]
                image_path = output_dir / f"image_{self.iteration}_{i}.png"
                image.save(image_path)
                self.generated_this_iteration.append((prompt, str(image_path)))
                print(f"Generated image for '{prompt}' at {image_path}")
            except Exception as e:
                print(f"Error generating image for prompt '{prompt}': {e}")
                # Optionally, handle errors like skipping this prompt or retrying

        print(f"✅ Generated {len(self.generated_this_iteration)} images for iteration {self.iteration + 1}.")
        self.next(self.evaluate_images)

    @step
    def evaluate_images(self):
        """
        Evaluates the generated images against their prompts using Google Gemma-3-4b-it
        and selects top-scoring prompts for the next iteration.
        """
        print(f"🧠 Scoring images with Google Gemma-3-4b-it (Iteration {self.iteration + 1}/{self.num_iterations})...")

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {device}")

        # Load processor and model for Gemma-3-4b-it
        processor = AutoProcessor.from_pretrained("google/gemma-3-4b-it")
        model = AutoModelForImageTextToText.from_pretrained(
            "google/gemma-3-4b-it",
            torch_dtype=torch.float16 if device.type == "cuda" else torch.float32 # Use float16 on GPU
        ).to(device)

        def score_prompt_image(prompt, image_path):
            """
            Scores how well an image matches a prompt using the Gemma model.
            Returns a score (0.0 or 1.0) based on a 'yes' or 'no' answer.
            """
            try:
                image = Image.open(image_path).convert("RGB")
                # Construct VQA prompt for Gemma
                question = f"Does this image accurately represent the prompt: '{prompt}'? Answer only with 'yes' or 'no'."
                inputs = processor(images=image, text=question, return_tensors="pt").to(device)
                output = model.generate(**inputs, max_new_tokens=5) # Limit output length for answer
                answer = processor.tokenizer.decode(output[0], skip_special_tokens=True).strip().lower()

                # Simple scoring: 1.0 if 'yes' is in the answer, 0.0 otherwise
                score = 1.0 if "yes" in answer else 0.0
                print(f"Prompt: '{prompt}', Image: {image_path}, Answer: '{answer}', Score: {score}")
                return score
            except Exception as e:
                print(f"Error scoring image {image_path} for prompt '{prompt}': {e}")
                return 0.0 # Return 0.0 on error

        scored_images_this_iteration = []
        for prompt, img_path in self.generated_this_iteration:
            score = score_prompt_image(prompt, img_path)
            scored_images_this_iteration.append((prompt, img_path, score))
            self.all_generated_data.append((prompt, img_path, score, self.iteration))

        # Sort images by score in descending order
        scored_images_this_iteration.sort(key=lambda x: x[2], reverse=True)
        print(f"✅ Scored {len(scored_images_this_iteration)} images for iteration {self.iteration + 1}.")

        # Select top K prompts for the next iteration
        self.selected_prompts = [
            prompt for prompt, _, _ in scored_images_this_iteration[:self.top_k_prompts_for_next_iteration]
        ]
        print(f"Selected {len(self.selected_prompts)} top prompts for the next iteration.")

        self.iteration += 1 # Increment iteration counter

        # Decide next step based on iteration count
        if self.iteration < self.num_iterations:
            print(f"🔄 Continuing to next iteration ({self.iteration + 1}/{self.num_iterations})...")
            self.next(self.select_prompts) # Loop back for another round
        else:
            print(f"🎉 Completed all {self.num_iterations} iterations.")
            self.next(self.upload_to_hf_hub) # Proceed to upload

    @step
    def upload_to_hf_hub(self):
        """
        Collects all generated data across iterations and uploads it as a dataset
        to Hugging Face Hub.
        """
        print("🚀 Preparing to upload dataset to Hugging Face Hub...")

        # Ensure you are logged in to Hugging Face Hub
        # You can do this by running `huggingface-cli login` in your terminal
        # before running this Metaflow flow.
        try:
            api = HfApi()
            user_info = api.whoami()
            print(f"Logged in to Hugging Face Hub as: {user_info['name']}")
        except Exception as e:
            print(f"Error logging into Hugging Face Hub. Please run `huggingface-cli login` in your terminal. Error: {e}")
            # Exit gracefully or raise an error if login is critical
            raise

        # Prepare data for Hugging Face Dataset
        prompts = [item[0] for item in self.all_generated_data]
        image_paths = [item[1] for item in self.all_generated_data]
        scores = [item[2] for item in self.all_generated_data]
        iterations = [item[3] for item in self.all_generated_data]

        # Create a dictionary for the dataset
        data_dict = {
            "prompt": prompts,
            "image": [Image.open(p).convert("RGB") for p in image_paths], # Load PIL Images
            "score": scores,
            "iteration": iterations
        }

        # Create a Hugging Face Dataset object
        # The 'image' column will automatically be handled as an ImageFeature
        dataset = Dataset.from_dict(data_dict)
        dataset = dataset.cast_column("image", ImageFeature()) # Explicitly cast to ImageFeature

        print(f"Created Hugging Face Dataset with {len(dataset)} examples.")
        print(dataset) # Print dataset structure

        # Push the dataset to Hugging Face Hub
        try:
            print(f"Uploading dataset to {self.hf_dataset_name}...")
            dataset.push_to_hub(self.hf_dataset_name, private=False) # Set private=True if you want it private
            print(f"✅ Dataset successfully uploaded to https://huggingface.co/datasets/{self.hf_dataset_name}")
        except Exception as e:
            print(f"Error uploading dataset to Hugging Face Hub: {e}")
            # You might want to add retry logic or more robust error handling here.
            raise

        self.next(self.end)

    @step
    def end(self):
        """
        Final step of the pipeline, prints a summary of the top-scoring generated images.
        """
        print("🍾 Done! Synthetic image data pipeline completed.")
        # Sort all collected data by score to show overall top images
        final_scored_data = sorted(self.all_generated_data, key=lambda x: x[2], reverse=True)
        print("\nOverall Top 5 Generated Images by Gemma Score:")
        for i, (prompt, path, score, iteration) in enumerate(final_scored_data[:5]):
            print(f"{i+1}. [Iter {iteration+1}] '{prompt}' → {path} [score: {score:.3f}]")

if __name__ == "__main__":
    SyntheticImageFlow()